# Ders 13 - Ajan Hafızası


## Kurulum

Bu not defteri, **Microsoft Ajan Çerçevesi** (MAF) kullanarak **kalıcı hafızaya** sahip bir seyahat rezervasyon ajanının nasıl oluşturulacağını gösterir.

Farklı ajan hafızası türlerinin — çalışma, kısa süreli ve uzun süreli — bir ajanın bilgileri konuşmalar arasında nasıl sakladığını ve kullandığını nasıl etkilediğini öğreneceksiniz.

**Önkoşullar:**
- Dağıtılmış sohbet modeli içeren bir Azure AI Foundry projesi (örneğin `gpt-4o-mini`).
- Azure CLI ile giriş yapılmış — terminalinizde `az login` komutunu çalıştırın.
- `AZURE_AI_PROJECT_ENDPOINT` — Azure AI Foundry proje uç noktanız.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — dağıtılan modelinizin adı.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Ajan Bellek Türleri

Yapay zeka ajanları, her biri farklı bir amaca hizmet eden çeşitli bellek türlerinden yararlanabilir:

### Çalışma Belleği
Konuşma dizisi — tek bir oturumda değiş tokuş edilen mesajlar. Ajan, aynı dizideki önceki mesajlara geri dönerek tutarlılığı sürdürebilir. MAF'te bu, **`agent.create_session()`** ile oluşturulur ve bir `AgentSession` döner.

### Kısa Süreli Bellek
Bir görev veya oturum süresince kalan ancak kalıcı olarak depolanmayan bilgiler. Örneğin, ajan çok aşamalı bir planlama konuşması sırasında bilgileri biriktirip bunları son bir rota oluşturmak için kullanabilir.

### Uzun Süreli Bellek
**Oturumlar arasında** kalan tercih ve gerçekler. Geri dönen bir kullanıcı, diyet kısıtlamalarını veya seyahat tarzını tekrar etmek zorunda kalmamalıdır. Uzun süreli bellek genellikle harici bir depolama — bir veritabanı, dosya veya vektör indexi — tarafından desteklenir ve araçlar aracılığıyla ajana sunulur.


## Oturumlarla Çalışan Bellek

Belleğin en basit şekli, konuşma oturumudur. Aynı oturumu ( `agent.create_session()` ile oluşturulan) ardışık `agent.run()` çağrılarına ilettiğinizde, ajan o konuşmanın tamamı geçmişini görür ve önceki detayları hatırlayabilir.

Bir seyahat acentesi oluşturalım ve çalışan belleği gösterelim.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Temsilci bütçeyi doğru hatırladı çünkü her iki mesaj da aynı oturumu paylaşıyor. Bu **çalışma belleği** — yalnızca oturumun süresi boyunca var olur.

### Yeni bir ileti dizisi ile ne olur?

Eğer **yeni** bir oturum oluşturursak, temsilcinin önceki konuşmanın hiçbir anısı olmaz:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Uzun Vadeli Bellek Deseni

Kullanıcının tercihlerini **oturumlar arasında** hatırlamak için, konuşma dizisinin dışında yaşayan kalıcı bir depo gerekir. Ajan, bu depoya erişmek için **araçlar** — bilgiyi kaydetmek ve almak için çağırabileceği fonksiyonlar — kullanır.

Aşağıda basit bir bellek içi tercih deposu (üretimde bunu bir veritabanı veya vektör dizini ile desteklersiniz) uyguluyoruz ve ajanın kullanabileceği araçlar olarak sunuyoruz.

### Mimari
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Senaryo 1 — İlk kez kullanıcı yıldönümü gezisi rezervasyonu yapıyor

Sarah ilk kez ziyarette bulunuyor. Temsilci, araçlar aracılığıyla onun tercihlerini kaydetmeli ve otel önerilerinde bu tercihleri kullanmalıdır.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Senaryo 2 — Sarah haftalar sonra geri döner

Sarah, **bambaşka bir konu** başlatır (yeni bir oturum simülasyonu). Çalışma belleği boştur, ancak uzun süreli tercih deposunda hala onun bilgileri bulunmaktadır. Temsilci bu bilgileri almalı ve önerileri kişiselleştirmek için kullanmalıdır.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Özet

Bu derste üç tür ajan belleği ve bunların Microsoft Agent Framework ile nasıl uygulanacağını öğrendiniz:

| Bellek Türü | MAF Mekanizması | Ömür |
|---|---|---|
| **Çalışma** | `agent.create_session()` | Tek oturum |
| **Kısa süreli** | Bir iş parçacığı içindeki birikmiş bağlam | Tek görev / oturum |
| **Uzun süreli** | `@tool` işlevleri ile erişilen dış depo | Oturumlar arasında |

### Temel Çıkarımlar
1. **`agent.create_session()`** çalışma belleği sağlar — ajan bir oturum içindeki tüm konuşma geçmişini görür.
2. **Yeni oturumlar bağlamı kaybeder** — uzun süreli bellek olmadan ajan önceki konuşmaları hatırlayamaz.
3. **`@tool` işlevleri** aradaki boşluğu kapatır — ajan bilgi kaydetmesine ve kalıcı depodan bilgi almasına izin verir.
4. **Kişiselleştirme zamanla gelişir** — tercihlerin daha fazla saklanması, ajanın önerilerini iyileştirir.

### Gerçek Dünya Uygulamaları
- **Müşteri Hizmetleri**: Müşteri geçmişini ve tercihlerini hatırlama
- **Kişisel Asistanlar**: Günler veya haftalar boyunca bağlamı koruma
- **Sağlık Hizmetleri**: Hasta bilgileri ve tercihlerini takip etme
- **E-ticaret**: Geçmişe dayalı kişiselleştirilmiş alışveriş

### Sonraki Adımlar
- Bellek içi sözlüğü bir veritabanı veya vektör deposu ile değiştirme (örneğin Azure AI Search)
- Zaman duyarlı bilgiler için bellek süresi sonu ekleme
- Paylaşılan bellek ile çoklu ajan sistemleri oluşturma
- Bilgi grafiği destekli bellek için Cognee defterini keşfetme


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Feragatname**:
Bu belge, AI çeviri hizmeti [Co-op Translator](https://github.com/Azure/co-op-translator) kullanılarak çevrilmiştir. Doğruluk için çaba gösterilse de, otomatik çevirilerin hata veya yanlışlık içerebileceğini lütfen unutmayınız. Orijinal dildeki belge otoriter kaynak olarak kabul edilmelidir. Kritik bilgiler için profesyonel insan çevirisi tavsiye edilir. Bu çevirinin kullanımı sonucunda oluşabilecek yanlış anlamalar veya yanlış yorumlardan sorumlu değiliz.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
